# 🛰️ Satellite Data Verification Notebook

This notebook verifies that the flood detection system is receiving **actual satellite imagery** from Google Earth Engine.

We will:
1. Download RAW Sentinel-1 SAR data
2. Visualize the actual pixel values
3. Compare dry season vs monsoon season
4. Verify flood detection logic

## 1. Setup & Imports

In [ ]:
import ee
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Initialize Earth Engine - USE YOUR PROJECT ID
ee.Initialize(project='caramel-pulsar-475810-e7')

print("✅ Earth Engine initialized successfully!")

## 2. Define Test Area

We'll test the **Sylhet - Surma River Basin** where we detected 37% flooding.

In [ ]:
# Test Area Configuration
TEST_AREA = {
    'name': 'Sylhet - Surma River Basin',
    'bounds': [91.85, 24.88, 91.95, 24.98],
}

# Date ranges
DRY_SEASON_START = '2024-01-01'
DRY_SEASON_END = '2024-03-31'

MONSOON_START = '2024-06-01'
MONSOON_END = '2024-09-30'

print(f"📍 Test Area: {TEST_AREA['name']}")
print(f"📅 Dry Season: {DRY_SEASON_START} to {DRY_SEASON_END}")
print(f"📅 Monsoon: {MONSOON_START} to {MONSOON_END}")

## 3. Download SAR Data Function

In [ ]:
def download_sar_data(bounds, start_date, end_date, period_name):
    print(f"\n{'='*60}")
    print(f"📥 Downloading: {period_name}")
    print(f"{'='*60}")
    
    geometry = ee.Geometry.Rectangle(bounds)
    
    collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .select(['VV', 'VH']))
    
    count = collection.size().getInfo()
    print(f"\n📊 Found {count} Sentinel-1 images")
    
    if count == 0:
        return None, None, None
    
    # Get image dates
    image_list = collection.toList(min(10, count))
    dates = []
    print(f"\n📅 Image dates:")
    for i in range(min(5, count)):
        img = ee.Image(image_list.get(i))
        timestamp = img.get('system:time_start').getInfo()
        date = datetime.fromtimestamp(timestamp / 1000).strftime('%Y-%m-%d')
        dates.append(date)
        print(f"   {i+1}. {date}")
    
    image = collection.median()
    
    # Statistics
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True).combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=geometry, scale=30, maxPixels=1e9
    ).getInfo()
    
    print(f"\n📈 VV Statistics: Mean={stats.get('VV_mean', 0):.2f}, Min={stats.get('VV_min', 0):.2f}, Max={stats.get('VV_max', 0):.2f}")
    
    # Download pixels
    try:
        sample = image.sampleRectangle(region=geometry, defaultValue=-999).getInfo()
        vv = np.array(sample['properties']['VV'])
        vh = np.array(sample['properties']['VH'])
        vv[vv == -999] = np.nan
        vh[vh == -999] = np.nan
        print(f"\n✅ Downloaded: {vv.shape}")
        return np.stack([vv, vh], axis=-1), stats, dates
    except Exception as e:
        print(f"❌ Error: {e}")
        return None, None, None

## 4. Download Both Seasons

In [ ]:
# Download DRY season
dry_data, dry_stats, dry_dates = download_sar_data(
    TEST_AREA['bounds'], DRY_SEASON_START, DRY_SEASON_END, "DRY SEASON"
)

In [ ]:
# Download MONSOON season
monsoon_data, monsoon_stats, monsoon_dates = download_sar_data(
    TEST_AREA['bounds'], MONSOON_START, MONSOON_END, "MONSOON SEASON"
)

## 5. 🔍 KEY VISUALIZATION - Raw SAR Images

In [ ]:
vv_dry = dry_data[:, :, 0]
vv_monsoon = monsoon_data[:, :, 0]
vh_dry = dry_data[:, :, 1]
vh_monsoon = monsoon_data[:, :, 1]

print(f"Image shapes - Dry: {vv_dry.shape}, Monsoon: {vv_monsoon.shape}")

In [ ]:
# Main visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(f'RAW Sentinel-1 SAR: {TEST_AREA["name"]}', fontsize=16, fontweight='bold')

# Row 1: VV comparison
im1 = axes[0, 0].imshow(vv_dry, cmap='gray', vmin=-25, vmax=-5)
axes[0, 0].set_title(f'DRY VV\nMean: {np.nanmean(vv_dry):.2f} dB')
axes[0, 0].axis('off')
plt.colorbar(im1, ax=axes[0, 0], fraction=0.046)

im2 = axes[0, 1].imshow(vv_monsoon, cmap='gray', vmin=-25, vmax=-5)
axes[0, 1].set_title(f'MONSOON VV\nMean: {np.nanmean(vv_monsoon):.2f} dB')
axes[0, 1].axis('off')
plt.colorbar(im2, ax=axes[0, 1], fraction=0.046)

vv_diff = vv_monsoon - vv_dry
im3 = axes[0, 2].imshow(vv_diff, cmap='RdBu_r', vmin=-10, vmax=10)
axes[0, 2].set_title(f'CHANGE\nMean: {np.nanmean(vv_diff):.2f} dB\n(Blue=More Water)')
axes[0, 2].axis('off')
plt.colorbar(im3, ax=axes[0, 2], fraction=0.046)

# Row 2: Water detection
water_thresh = -15
dry_water = (vv_dry < water_thresh).astype(float)
monsoon_water = (vv_monsoon < water_thresh).astype(float)
new_flood = np.maximum(0, monsoon_water - dry_water)

dry_pct = np.nanmean(dry_water) * 100
monsoon_pct = np.nanmean(monsoon_water) * 100
flood_pct = np.nanmean(new_flood) * 100

axes[1, 0].imshow(dry_water, cmap='Blues')
axes[1, 0].set_title(f'DRY Water\n{dry_pct:.1f}%')
axes[1, 0].axis('off')

axes[1, 1].imshow(monsoon_water, cmap='Blues')
axes[1, 1].set_title(f'MONSOON Water\n{monsoon_pct:.1f}%')
axes[1, 1].axis('off')

axes[1, 2].imshow(new_flood, cmap='Reds')
axes[1, 2].set_title(f'NEW FLOODING\n{flood_pct:.1f}%')
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig('verification_result.png', dpi=150)
plt.show()

## 6. Histogram Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(vv_dry.flatten(), bins=50, alpha=0.6, label=f'Dry (mean: {np.nanmean(vv_dry):.1f} dB)', color='orange')
ax.hist(vv_monsoon.flatten(), bins=50, alpha=0.6, label=f'Monsoon (mean: {np.nanmean(vv_monsoon):.1f} dB)', color='blue')
ax.axvline(x=-15, color='red', linestyle='--', linewidth=2, label='Water threshold (-15 dB)')

ax.set_xlabel('VV Backscatter (dB)')
ax.set_ylabel('Pixel Count')
ax.set_title('VV Value Distribution - Dry vs Monsoon')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Sample Pixel Values

In [ ]:
h, w = vv_dry.shape
ch, cw = h // 2, w // 2

print(f"Image size: {h} x {w} pixels")
print(f"\nCenter 5x5 pixel values (dB):")
print(f"\nDRY VV:")
print(vv_dry[ch-2:ch+3, cw-2:cw+3].round(2))
print(f"\nMONSOON VV:")
print(vv_monsoon[ch-2:ch+3, cw-2:cw+3].round(2))
print(f"\nCHANGE:")
print((vv_monsoon[ch-2:ch+3, cw-2:cw+3] - vv_dry[ch-2:ch+3, cw-2:cw+3]).round(2))

## 8. Final Summary

In [ ]:
print("="*70)
print("📊 VERIFICATION SUMMARY")
print("="*70)
print(f"""
📍 Area: {TEST_AREA['name']}
📏 Size: {vv_dry.shape[0]} x {vv_dry.shape[1]} pixels

                    DRY             MONSOON
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
VV Mean:            {np.nanmean(vv_dry):.2f} dB        {np.nanmean(vv_monsoon):.2f} dB
VV Std:             {np.nanstd(vv_dry):.2f} dB        {np.nanstd(vv_monsoon):.2f} dB
Water %:            {dry_pct:.1f}%            {monsoon_pct:.1f}%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📈 New Flooding: {flood_pct:.1f}%
📈 VV Change: {np.nanmean(vv_diff):.2f} dB

✅ QUALITY CHECKS:
""")

if -30 < np.nanmean(vv_dry) < 0:
    print("   ✅ VV values in expected range")
else:
    print("   ❌ VV values unexpected!")
    
if np.nanstd(vv_dry) > 1:
    print("   ✅ Image has texture (not flat)")
else:
    print("   ⚠️ Image appears uniform")

if vv_dry.shape[0] > 50:
    print(f"   ✅ Good resolution ({vv_dry.shape[0]}x{vv_dry.shape[1]})")
else:
    print(f"   ⚠️ Low resolution")

print("\n" + "="*70)

## 🔄 Test Another Area

In [ ]:
# Change TEST_AREA to any of these and re-run
OTHER_AREAS = {
    'jamuna_river': [89.70, 24.30, 89.85, 24.45],
    'dhaka_east': [90.40, 23.75, 90.50, 23.85],
    'sundarbans': [89.60, 22.10, 89.75, 22.25],
}

print("To test another area, change TEST_AREA['bounds'] to one of:")
for name, bounds in OTHER_AREAS.items():
    print(f"  • {name}: {bounds}")